# H3: Post-Variational Strategies on Non-Convex QML Landscapes

**Hypothesis**: Post-variational methods (observable construction, ansatz expansion) consistently achieve higher test accuracy than fully variational training on non-convex benchmark tasks.

**References**: arXiv:2307.10560, arXiv:2403.07059

In [ ]:
import pennylane as qml
from pennylane import numpy as np
from itertools import combinations
from sklearn.datasets import make_moons, make_classification
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
import json
import os
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)
qml.numpy.random.seed(42)

In [ ]:
N_WIRES = 4
N_EPOCHS = 30
LR = 0.1

def feature_map(features, n_wires=N_WIRES):
    for i in range(n_wires):
        qml.Hadamard(i)
    for i in range(len(features)):
        if i % 2:
            qml.AngleEmbedding(features=features[i], wires=range(n_wires), rotation="Z")
        else:
            qml.AngleEmbedding(features=features[i], wires=range(n_wires), rotation="X")

def ansatz(params, n_wires=N_WIRES):
    for i in range(n_wires):
        qml.RY(params[i], wires=i)
    for i in range(n_wires):
        qml.CNOT(wires=[(i - 1) % n_wires, i % n_wires])
    for i in range(n_wires):
        qml.RY(params[i + n_wires], wires=i)

def local_pauli_group(qubits, locality):
    result = []
    def _gen(idx, paulis, current):
        if idx == qubits:
            result.append(current)
            return
        _gen(idx + 1, paulis, current + "I")
        if paulis < locality:
            _gen(idx + 1, paulis + 1, current + "X")
            _gen(idx + 1, paulis + 1, current + "Y")
            _gen(idx + 1, paulis + 1, current + "Z")
    _gen(0, 0, "")
    return result

In [ ]:
def run_variational(X_train, y_train, X_test, y_test):
    dev = qml.device("default.qubit", wires=N_WIRES)
    
    @qml.qnode(dev)
    def circuit(params, features):
        feature_map(features)
        ansatz(params)
        return qml.expval(qml.PauliZ(0))
    
    weights = 0.01 * np.random.randn(2 * N_WIRES)
    bias = 0.0
    
    for epoch in range(N_EPOCHS):
        for i in range(len(X_train)):
            pred = circuit(weights, X_train[i]) + bias
            loss = (pred - y_train[i]) ** 2
            weights -= LR * 2 * (pred - y_train[i])
            bias -= LR * 2 * (pred - y_train[i])
    
    preds = [np.sign(circuit(weights, x) + bias) for x in X_test]
    return float(accuracy_score(y_test, preds))

def run_observable(X_train, y_train, X_test, y_test, locality=2):
    dev = qml.device("default.qubit", wires=N_WIRES)
    measurements = local_pauli_group(N_WIRES, locality)
    
    @qml.qnode(dev)
    def circuit(features):
        feature_map(features)
        return [qml.expval(qml.pauli.string_to_pauli_word(m)) for m in measurements]
    
    X_train_t = np.array([circuit(x) for x in X_train])
    X_test_t = np.array([circuit(x) for x in X_test])
    
    clf = MLPClassifier(early_stopping=True, max_iter=200, random_state=42)
    clf.fit(X_train_t, y_train)
    return float(clf.score(X_test_t, y_test))

def run_mlp(X_train, y_train, X_test, y_test):
    clf = MLPClassifier(early_stopping=True, max_iter=200, random_state=42)
    clf.fit(X_train, y_train)
    return float(clf.score(X_test, y_test))

In [ ]:
datasets = {
    "moons_easy": make_moons(n_samples=200, noise=0.05, random_state=42),
    "moons_hard": make_moons(n_samples=200, noise=0.3, random_state=42),
    "blobs": make_classification(n_samples=200, n_features=4, n_informative=2,
                                  n_redundant=0, n_clusters_per_class=1, random_state=42),
}

results = {}
for dset_name, (X, y) in datasets.items():
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
    y_train = (y_train - 0.5) * 2
    y_test = (y_test - 0.5) * 2
    
    print(f"\n--- {dset_name} ---")
    acc_var = run_variational(X_train, y_train, X_test, y_test)
    acc_obs1 = run_observable(X_train, y_train, X_test, y_test, locality=1)
    acc_obs2 = run_observable(X_train, y_train, X_test, y_test, locality=2)
    acc_mlp = run_mlp(X_train, y_train, X_test, y_test)
    
    results[dset_name] = {
        "variational": acc_var,
        "observable_1_local": acc_obs1,
        "observable_2_local": acc_obs2,
        "MLP_classical": acc_mlp,
    }
    print(f"  Variational:       {acc_var:.4f}")
    print(f"  Observable (1-loc): {acc_obs1:.4f}")
    print(f"  Observable (2-loc): {acc_obs2:.4f}")
    print(f"  MLP (classical):    {acc_mlp:.4f}")

os.makedirs('results/H3', exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
with open(f'results/H3/run_{ts}.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f"\nResults saved to results/H3/run_{ts}.json")

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 5))
strategies = ['variational', 'observable_1_local', 'observable_2_local', 'MLP_classical']
colors = ['C0', 'C1', 'C2', 'C3']
x = np.arange(len(datasets))
w = 0.2

for i, s in enumerate(strategies):
    vals = [results[d][s] for d in datasets]
    ax.bar(x + i * w, vals, w, label=s.replace('_', ' ').title(), color=colors[i])

ax.set_xticks(x + w * 1.5)
ax.set_xticklabels(list(datasets.keys()), rotation=15)
ax.set_ylabel('Test Accuracy')
ax.set_title('H3: Post-Variational Strategies Benchmark')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
os.makedirs('results/H3', exist_ok=True)
plt.savefig('results/H3/plot.png', dpi=120)
plt.show()